In [1]:
from schemas.event import Event, EventClass
from annotation.tools.event_classifier import hierarchical_event_classification, extract_event_name, extract_event_agents
from utils.loader import load_csv, load_json_folder, save_structured_folktale
from annotation.tools.place_extractor import extract_places
from annotation.tools.agent_extractor import extract_agents
from annotation.tools.genre_extractor import extract_genre
from annotation.tools.event_chunking import extract_story_segments
from annotation.tools.event_extractor import extract_event_elements
from annotation.tools.object_extractor import extract_objects
from annotation.tools.relationship_extractor import extract_relationships
from utils.regex_utils import clean_regex, title_case_to_snake_case
from utils.models import get_llm
from schemas.folktale import Folktale
from pandas import DataFrame
from loguru import logger
from tqdm import tqdm
import re
import os
import json

In [2]:
from schemas.folktale import Genre, GenreClass
from schemas.object import Object
from schemas.place import Place
from schemas.agent import Agent
from schemas.relationship import Relationship

In [3]:
import pydantic
print(pydantic.__version__)

2.12.5


In [4]:
def get_batch(df: DataFrame, start: int, size: int):
    if start < 0:
        raise ValueError("start must be >= 0")

    if size < 0:
        raise ValueError("size must be >= 0")

    return df.iloc[start:start + size]

model = get_llm(0.7)

In [5]:
data_dir = "./data"
processed_dir = os.path.join(data_dir, "processed")
folktales_path = os.path.join(processed_dir, "folk_tales_deduplicated.csv")
folktales_df = load_csv(folktales_path)
selected_folktales_df = get_batch(folktales_df, 0, 5)

metadata_dir = "./metadata"
structure_dir = os.path.join(metadata_dir, "structure")
collections_dir = os.path.join(metadata_dir, "collections")
entities_dir = os.path.join(metadata_dir, "entities")

out_dir = "./out_prueba"
os.makedirs(out_dir, exist_ok=True)

structures = load_json_folder(structure_dir)
collections = load_json_folder(collections_dir)
entities = load_json_folder(entities_dir)

n_folktales = len(selected_folktales_df)

i = 1
row = selected_folktales_df.iloc[i]

In [8]:
load = True
text = row["text"]
url = row["source"]
nation = row["nation"]
title = row["title"]

filename = re.sub(clean_regex, "", title)
filename = title_case_to_snake_case(filename)

json_path = os.path.join(out_dir, f"{i}_{filename}_annotation.json")

if load and os.path.exists(json_path):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    genre = GenreClass(data["genre"])
    objects = [Object(**o) for o in data["objects"]]
    places = [Place(**p) for p in data["places"]]
    agents = [Agent(**a) for a in data["agents"]]
    relationships = [Relationship(**r) for r in data["relationships"]]
    story_segments = data.get("story_segments", [])


else:
    logger.debug(f"Starting annotation: '{title}' (idx={i})")

    genre = extract_genre(model, text, collections["genre"])
    objects = extract_objects(model, text, entities["object"])
    places = extract_places(model, text, entities["place"])
    agents = extract_agents(
        model, text, places,
        structures["role"],
        collections["trait"]
    )
    relationships = extract_relationships(model, text, agents)
    story_segments = extract_story_segments(model, text)

    # Serialización
    data = {
        "genre": genre.model_dump(),
        "objects": [o.model_dump() for o in objects],
        "places": [p.model_dump() for p in places],
        "agents": [a.model_dump() for a in agents],
        "relationships": [r.model_dump() for r in relationships],
        "story_segments": story_segments,
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)



In [ ]:
logger.remove()


In [10]:
events = []
for j, segment in tqdm(enumerate(story_segments), desc=f"Events ({title})", leave=False):
    objects_ids, place_id = extract_event_elements(model, title, segment, objects, places)

    event_type, final_thoughts = hierarchical_event_classification(
        model=model,
        event_index=j,
        story_segments=story_segments,
        taxonomy_tree=structures["function"],
        n_rounds=3,
        verbose=False
    )
    
    if event_type is None:
        event_type = EventClass.EVENT
    name = extract_event_name(model, event_type, segment, final_thoughts)
    event_agents = extract_event_agents(model, segment, final_thoughts, agents, title)
    
    # u = uuid.uuid4()
    # fixed = f"place_{u.hex}"

    event = Event(
        type=event_type,
        name=name,
        description=segment,
        agents=event_agents,
        objects=objects_ids,
        place=place_id,
        thoughts=final_thoughts
    )

    events.append(event)

folktale = Folktale(
    url=url,
    nation=nation,
    title=title,
    genre=genre,
    agents=agents,
    relationships=relationships,
    places=places,
    objects=objects,
    events=events
)

filename = re.sub(clean_regex, "", title)
filename = title_case_to_snake_case(filename)
path = os.path.join(out_dir, f"{i}_{filename}")
save_structured_folktale(folktale, path)

